In [ ]:
import numpy as np
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import evaluate

In [ ]:
print("Đang tải dataset...")
dataset = load_dataset("imdb")

In [ ]:
train_dataset = dataset["train"].shuffle(seed=42).select(range(2000))
eval_dataset = dataset["test"].shuffle(seed=42).select(range(500))

In [ ]:
model_checkpoint = "bert-base-uncased" # Dùng Bert gốc cho ổn định nhất
print(f"Đang tải model: {model_checkpoint}...")

In [ ]:
try:
    tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
except:
    tokenizer = AutoTokenizer.from_pretrained(model_checkpoint, use_fast=False)

model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=2)

In [ ]:
def tokenize_function(examples):
    # max_length=256 giúp model đọc được câu dài hơn -> Accuracy cao hơn 128
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=256)

print("Đang xử lý dữ liệu (Tokenization)...")

# SỬA LỖI: Dùng đúng tên biến train_dataset đã khai báo ở trên
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_eval = eval_dataset.map(tokenize_function, batched=True)

In [ ]:
training_args = TrainingArguments(
    output_dir="bert_high_acc",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,             # Train 3 vòng
    weight_decay=0.01,
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none"
)

In [ ]:
metric = evaluate.load("accuracy")
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    compute_metrics=compute_metrics,
)

print("Bắt đầu Training...")
trainer.train()

In [ ]:
print("\nKết quả đánh giá trên tập test:")
metrics = trainer.evaluate()
print(f"\nAccuracy cuối cùng: {metrics['eval_accuracy']*100:.2f}%")

In [ ]:
print("\nTest thử model:")
text = "This movie is absolutely amazing and fantastic!" # Câu này nên là Positive
inputs = tokenizer(text, return_tensors="pt").to(model.device)
with torch.no_grad():
    logits = model(**inputs).logits
predicted_class_id = logits.argmax().item()
print(f"Text: '{text}'")
print(f"Label: {'POSITIVE' if predicted_class_id == 1 else 'NEGATIVE'}")